## Initialization

In [ ]:
#Imports
import math
import torch
import torchaudio
from copy import deepcopy
from torch import nn
from torch.utils.data import DataLoader
#from torchvision import transforms

#Custom modules
import cAudiotools
import cLogger
import cTransforms
import cModelManager
import cNNModules

MODEL_NAME = "CNN_Raw_2_VAD"
MODELS_DIR = r"output\models"
model_description = "CNN for VAD regression in speech using only raw audio as input."

#Define output paths
log = cLogger.Log("output\logs", prefix=MODEL_NAME)
modelMngr = cModelManager.ModelManager(f"{MODELS_DIR}\{MODEL_NAME}")
#model_dir = cModelManager.Directories.make_unique(f"output\models\{MODEL_NAME}")
log.log_property("model_name", MODEL_NAME)
log.log_property("model_description", model_description, show=False)
log.log_property("model_dir", modelMngr.model_directory, show=False)

#Torch properties and device
log.log_property("torch_version", torch.__version__)
log.log_property("torchaudio_version", torchaudio.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    log.log_property("device", "cuda")
    log.log_property("GPU_count", torch.cuda.device_count())
    log.log_property("GPU_device", torch.cuda.get_device_name(0))
else:
    log.log_property("device", "cpu")
    

## Define data parameters

In [ ]:
#Parameters:
loader_params = {
    "dataset_train": r"C:\Datasets\_compiled\msp-podcast-2\labels_train_VAD.csv",
    "dataset_train_dir": r"C:\Datasets\_compiled\msp-podcast-2\Train",
    "dataset_dev": r"C:\Datasets\_compiled\msp-podcast-2\labels_development_VAD.csv",
    "dataset_dev_dir": r"C:\Datasets\_compiled\msp-podcast-2\Development",
    "shuffle": True,
    "collate_function": cAudiotools.Collate.waveform_with_lengths,
    "data_transform": None,
    "target_transform": cTransforms.NormalizeMinus(1, 7)
}

training_params = {
    "epochs": 50,
    "batch_size": 16,
    "checkpoint_interval": 5,
    "checkpoint_before_training": True,
    "criterion_for_best": "val_avg_loss",
}

optimizer_params = {    
    "learning_rate": 1e-3,
    "adam_betas": (0.9, 0.999),
    "adam_epsilon": 1e-8,
}

#scheduler_params = {
#    "use_scheduler": True,
#    "scheduler": torch.optim.lr_scheduler.ReduceLROnPlateau(),
#}

audio_params = {
    "sample_rate": 16000
}

#spectrogram_params = {
#    "n_fft": 512,
#    "hop_length": 160,
#    "win_length": 400,
#    "window_fn": torch.hamming_window,
#    "normalized": True,
#    "power": 2
#}
#
#mel_params = {
#    "n_mels": 64,
#    "pad_mode": "constant",
#    "mel_scale": "htk",
#    "n_mfcc": 20
#}


## Define training parameters

In [17]:

#spectogram_transform = torchaudio.transforms.Spectrogram(
#    n_fft=spectrogram_params["n_fft"],
#    hop_length=spectrogram_params["hop_length"],
#    win_length=spectrogram_params["win_length"],
#    window_fn=spectrogram_params["window_fn"],
#    normalized=spectrogram_params["normalized"],
#    power=spectrogram_params["power"]
#    )
#
#melspectogram_transform = torchaudio.transforms.MelSpectrogram(
#    sample_rate=audio_params["sample_rate"],
#    n_fft=spectrogram_params["n_fft"],
#    win_length=spectrogram_params["win_length"],
#    hop_length=spectrogram_params["hop_length"],
#    window_fn=spectrogram_params["window_fn"],
#    power=spectrogram_params["power"],
#    n_mels=mel_params["n_mels"],
#    normalized=spectrogram_params["normalized"],
#    pad_mode=mel_params["pad_mode"],
#    mel_scale=mel_params["mel_scale"]
#)
#
#mfcc_transform = torchaudio.transforms.MFCC(
#    sample_rate=audio_params["sample_rate"],
#    n_mfcc=mel_params["n_mfcc"],
#    melkwargs={
#        "n_fft": spectrogram_params["n_fft"],
#        "win_length": spectrogram_params["win_length"],
#        "hop_length": spectrogram_params["hop_length"],
#        "window_fn": spectrogram_params["window_fn"],
#        "power": spectrogram_params["power"],
#        "n_mels": mel_params["n_mels"],
#        "normalized": spectrogram_params["normalized"],
#        "pad_mode": mel_params["pad_mode"],
#        "mel_scale": mel_params["mel_scale"]
#    }
#)
#loader_params["data_transform"] = None

log.log_properties("Loader", loader_params, show=False)
log.log_properties("Training", training_params, show=False)
log.log_properties("Audio", audio_params, show=False)
#log.log_properties("Spectrogram", spectrogram_params)
#log.log_properties("Mel Spectrogram", mel_params)

#Train set
msp_vad_train = cAudiotools.AudioDatasetVAD(
    loader_params["dataset_train"], 
    loader_params["dataset_train_dir"],
    transform=loader_params["data_transform"],
    target_transform=loader_params["target_transform"]
    )
train_dataloader = DataLoader(
    msp_vad_train,
    batch_size=training_params["batch_size"],
    shuffle=loader_params["shuffle"],
    collate_fn=loader_params["collate_function"],
    pin_memory=True 
    )

#Development (validation) set
msp_vad_dev = cAudiotools.AudioDatasetVAD(
    loader_params["dataset_dev"],
    loader_params["dataset_dev_dir"],
    transform=loader_params["data_transform"],
    target_transform=loader_params["target_transform"]
    )
dev_dataloader = DataLoader(
    msp_vad_dev,
    batch_size=training_params["batch_size"],
    shuffle=loader_params["shuffle"],
    collate_fn=loader_params["collate_function"],
    pin_memory=True 
    )


## Data check

In [18]:
log.log_message(f"Train samples: {len(msp_vad_train)}")
log.log_message(f"Dev samples: {len(msp_vad_dev)}")
log.log_message(f"Batch size: {training_params['batch_size']}")

sample_batch = next(iter(train_dataloader))
inputs, lengths, targets = sample_batch

log.log_message("Sample batch:")
log.log_message(f"Inputs Shape: {inputs.shape}")
log.log_message(f"Targets Shape: {targets.shape}")
log.log_message(f"Lengths Shape: {lengths.shape}")
log.log_message(f"Input range: Min={inputs.min():.2f}, Max={inputs.max():.2f}")
log.log_message(f"Output range: Min={targets.min():.2f}, Max={targets.max():.2f}")

Train samples: 169190
Dev samples: 34399
Batch size: 64
Sample batch:
Inputs Shape: torch.Size([64, 1, 184320])
Targets Shape: torch.Size([64, 3])
Lengths Shape: torch.Size([64])
Input range: Min=-1.00, Max=1.00
Output range: Min=-1.00, Max=1.00


## Define Architecture

In [19]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()

        def conv_block(in_channels, out_channels, kernel_size, stride):
            return nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size,
                          stride=stride, padding=kernel_size // 2),
                nn.GroupNorm(num_groups=8, num_channels=out_channels),
                nn.ReLU(),
            )

        self.frontend = nn.Sequential(
            # 16kHz ~10ms = 160
            nn.Conv1d(1, 32, kernel_size=400, stride=10, padding=200),
            nn.GroupNorm(8, 32),
            nn.ReLU(),
        )

        self.encoder = nn.Sequential(
            conv_block(32, 64, 15, 2),
            conv_block(64, 128, 15, 2),
            conv_block(128, 256, 15, 2),
        )
        
        self.masked_pool = cNNModules.MaskedAvgPool1D()

        self.regression_head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(p=0.2),
            nn.Linear(128, 3),
            nn.Tanh()
        )

    def forward(self, input, audio_lengths):
        features = self.frontend(input)
        features = self.encoder(features)

        frontend_steps = (audio_lengths // 10) + 1
        downsampled_lengths = (frontend_steps + 7) // 8
        downsampled_lengths = torch.clamp(downsampled_lengths, min=1, max=features.size(-1))

        pooled_embedding = self.masked_pool(features, downsampled_lengths)        
        logits = self.regression_head(pooled_embedding)
        return logits


model = NeuralNetwork().to(device)
log.log_property("model_structure", str(model))

loss_fn = nn.MSELoss()
log.log_property("loss_function", str(loss_fn))

optimizer = torch.optim.Adam(
    model.parameters(), 
    lr=optimizer_params["learning_rate"], 
    betas=optimizer_params["adam_betas"],
    eps=optimizer_params["adam_epsilon"]
    )

log.log_property("optimizer", str(optimizer))

model_structure: NeuralNetwork(
  (frontend): Sequential(
    (0): Conv1d(1, 32, kernel_size=(400,), stride=(10,), padding=(200,))
    (1): GroupNorm(8, 32, eps=1e-05, affine=True)
    (2): ReLU()
  )
  (encoder): Sequential(
    (0): Sequential(
      (0): Conv1d(32, 64, kernel_size=(15,), stride=(2,), padding=(7,))
      (1): GroupNorm(8, 64, eps=1e-05, affine=True)
      (2): ReLU()
    )
    (1): Sequential(
      (0): Conv1d(64, 128, kernel_size=(15,), stride=(2,), padding=(7,))
      (1): GroupNorm(8, 128, eps=1e-05, affine=True)
      (2): ReLU()
    )
    (2): Sequential(
      (0): Conv1d(128, 256, kernel_size=(15,), stride=(2,), padding=(7,))
      (1): GroupNorm(8, 256, eps=1e-05, affine=True)
      (2): ReLU()
    )
  )
  (masked_pool): MaskedAvgPool1D()
  (regression_head): Sequential(
    (0): Linear(in_features=256, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=128, out_features=3, bias=True)
    (4): Tanh

## Training

In [21]:
#Loop definitions
def train_loop(dataloader, model, loss_fn, optimizer, metrics_dict=None):
    model.train()
    size = len(dataloader.dataset)
    epoch_loss = 0
    for batch, (inputs, lengths, targets) in enumerate(dataloader):
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        lengths = lengths.to(device, non_blocking=True)

        pred = model(inputs, lengths)
        loss = loss_fn(pred, targets)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * inputs.size(0)
        batch_loss = loss.item()

        if batch % 100 == 0:
            current = batch * len(inputs)
            print(f"[{current:>6d}/{size:>6d}] batch loss: {batch_loss:>7f}")
    
    epoch_loss /= size
    if metrics_dict:
        metrics_dict["train_avg_loss"] = epoch_loss

def validation_loop(dataloader, model, loss_fn, metrics_dict=None):
    model.eval()
    size = len(dataloader.dataset)
    test_loss = 0

    with torch.no_grad():
        for inputs, lengths, targets  in dataloader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            lengths = lengths.to(device, non_blocking=True)

            pred = model(inputs, lengths)
            loss = loss_fn(pred, targets)
            test_loss += loss.item() * inputs.size(0)

    test_loss /= size
    if metrics_dict:
        metrics_dict["val_avg_loss"] = test_loss


#Set Model Manager
modelMngr.set_model(model, optimizer, training_params["criterion_for_best"])

#Training
epoch_metrics = {'train_avg_loss': math.inf, 'val_avg_loss': math.inf}
remaining_for_checkpoint = training_params["checkpoint_interval"]

if training_params["checkpoint_before_training"]:
        modelMngr.checkpoint(0, epoch_metrics)

log.track_time(True, message="Starting training.")
total_epochs = training_params["epochs"]

for epoch in range(total_epochs):
    log.log_message(f"Epoch {epoch + 1} of {total_epochs}...")
    
    train_loop(train_dataloader, model, loss_fn, optimizer, metrics_dict=epoch_metrics)
    validation_loop(dev_dataloader, model, loss_fn, metrics_dict=epoch_metrics)

    log.log_epoch(epoch + 1, epoch_metrics)
    log.save()
    
    #Checkpointing
    remaining_for_checkpoint -= 1
    if remaining_for_checkpoint == 0:
        modelMngr.checkpoint(epoch + 1, epoch_metrics)
        remaining_for_checkpoint = training_params["checkpoint_interval"]
    
    #Save best model
    modelMngr.check_best(epoch + 1, epoch_metrics)

log.log_elapsed_time(message="Training completed")
log.track_time(False, show=False)
log.log_properties("Last_epoch", epoch_metrics)
log.log_properties("Best_model", modelMngr.best_model_metrics)
log.save()
log.save_txt()

modelMngr.save_for_inference()
modelMngr.save_best(save_for_inference=True)


Checkpoint saved at output\models\CNN_Raw_2_VAD\run_2026_01_25-002322\checkpoints\checkpoint_epoch_0.tar
Time tracking reset.
Starting training.
Epoch 1 of 50...


KeyboardInterrupt: 